# 2 - Knowledge Graphs and Exact Memory

A knowledge graph stores knowledge as **triples** `(head, relation, tail)`. We build a tiny baseline graph in pure Python to show the standard operations and their one limit, then contrast it with the **real GMS store** (the rebuilt policy store), which makes membership a graded distance.

## The baseline knowledge graph (pure Python)

In [ ]:
triples = [
    ('overdraft', 'has_fee_amount',  '35'),
    ('overdraft', 'governed_by',     'reg_e'),
    ('dispute',   'has_window_days', '60'),
]
def query(h=None, r=None, t=None):
    return [tr for tr in triples
            if (h is None or tr[0]==h) and (r is None or tr[1]==r)
            and (t is None or tr[2]==t)]
print('about overdraft:', query(h='overdraft'))
# membership is binary: the graph says yes/no, not 'how wrong'
print(('overdraft','has_fee_amount','35') in triples,
      ('overdraft','has_fee_amount','50') in triples)

## The real GMS store: membership becomes a graded distance
We load the trained policy store and call its real primitives. `query_triples` returns asserted edges exactly (as the baseline did); `score_triple` returns a geodesic distance --- low for a committed fact, high for a fabricated one --- the graded signal the baseline lacked.

In [ ]:
import torch
from pathlib import Path
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

def _find_store(name="gms_policy_store_geode"):
    for p in [Path.cwd(), *Path.cwd().parents]:
        c = p / "beyond-prompt-and-pray" / "code" / "data" / name
        if c.exists():
            return c
    raise FileNotFoundError(name + " not found; run scripts/build_geode_rag_store.py")

STORE = _find_store()
store = GMSExpertStore(DocGMSConfig(store_path=str(STORE), ingest_mode="regex"),
                       device=torch.device("cpu"))
assert store.load(), "store failed to load"
print("loaded store:", len(store.adapter.relation_to_idx), "relations,",
      len(store.adapter.entity_to_idx), "entities")

In [ ]:
print('asserted edges:', store.query_triples(head='overdraft')[:3])
d_true  = store.score_triple('overdraft', 'has_fee_amount', '35.0')
d_false = store.score_triple('overdraft', 'has_fee_amount', '50.0')
print(f'score_triple 35.0 = {d_true:.3f}  (committed)')
print(f'score_triple 50.0 = {d_false:.3f}  (not committed)')

## Self-check

In [ ]:
assert query(h='overdraft')                              # baseline pattern query
assert ('overdraft','has_fee_amount','35') in triples
assert d_true < d_false                                  # GMS grades membership
print('OK - knowledge graphs & exact memory')